# 01. Data Cleaning & Preparation

**Project Objective**: Predict residential real estate sale prices in **Capital Federal (CABA)**, Argentina.

**Notebook Scope**:
1. **Ingestion & Profiling**: Load raw listing dataset (`ar_properties_crude.csv`).
2. **Scope & Currency Unification**:
   - Retain residential properties (*Departamento*, *Casa*, *PH*) under *Venta* in *Capital Federal*.
   - **ARS to USD Conversion**: Convert listings quoted in Argentine Pesos (ARS) using the 2019 mean exchange rate of **1 USD = 50 ARS**.
   - **USD Inflation Adjustment**: Add a 30% cumulative US Dollar inflation adjustment (`price_usd_infl_adj = price_usd * 1.30`) to model in constant 2026 purchasing power dollars alongside nominal market prices.
3. **Location Normalization**: Rename geographic taxonomy (`l1` -> `country`, `l2` -> `region`, `l3` -> `barrio`, `l4` -> `sub_barrio`) and filter by CABA bounding box.
4. **Anti-Leakage Deduplication**: Remove exact ID duplicates and content-level duplicate listings across realtors.
5. **Physical Consistency**: Reconcile surface_total >= surface_covered, impute studio apartment (*monoambiente*) bedrooms, handle numeric missingness.
6. **Domain Fences**: Trim realistic real estate market outliers for price, surface, and price per m² ($500 to $8,000 USD/m²).
7. **Type Optimization**: Cast categorical and temporal features, handle active listing sentinel dates (`9999-12-31` -> `NaT`), and drop redundant constant columns.
8. **Export Handoff**: Save interim cleaned data to Parquet (`properties_cleaned.parquet`) for `02_eda_exploration.ipynb` and CSV fallback.


In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
print('Environment initialized successfully.')


Environment initialized successfully.


### 1. Ingestion & Initial Profiling
Load raw listings and inspect initial dimensionality and columns.


In [2]:
raw_data_path = Path('../data/raw/ar_properties_crude.csv')
print(f'Loading raw data from: {raw_data_path}')

df = pd.read_csv(raw_data_path)
print(f'Initial dataset shape: {df.shape[0]:,} rows, {df.shape[1]} columns')
df.head(3)


Loading raw data from: ..\data\raw\ar_properties_crude.csv
Initial dataset shape: 1,000,000 rows, 25 columns


### 2. Scope, Currency Unification & Inflation Adjustment
To build an accurate prediction model for residential properties in the capital city:
- **Country & Region**: Keep only `l1 == 'Argentina'` and `l2 == 'Capital Federal'` (CABA).
- **Operation**: Keep `operation_type == 'Venta'` (sales only).
- **Property Types**: Focus on residential living properties (`departamento`, `casa`, `ph`).
- **Currency Unification**: Convert listings quoted in `ARS` using the 2019 mean exchange rate of **1 USD = 50 ARS**.
- **Inflation Adjustment**: Engineer `price_usd_infl_adj = price_usd * 1.30` representing cumulative ~30% US Dollar inflation from 2019 to 2026.


In [3]:
valid_property_types = ['departamento', 'casa', 'ph']

# 1. Scope filter
df = df[
    (df['l1'] == 'Argentina') &
    (df['l2'] == 'Capital Federal') &
    (df['operation_type'] == 'Venta') &
    (df['currency'].isin(['USD', 'ARS'])) &
    (df['property_type'].str.lower().isin(valid_property_types))
].copy()

# 2. Convert ARS to USD using 2019 benchmark rate (1 USD = 50 ARS)
ARS_EXCHANGE_RATE = 50.0
df['price_usd'] = df['price'].copy()
mask_ars = df['currency'] == 'ARS'
df.loc[mask_ars, 'price_usd'] = df.loc[mask_ars, 'price'] / ARS_EXCHANGE_RATE

# 3. 30% USD Inflation Adjustment (2019 to 2026 constant dollars)
USD_INFLATION_FACTOR = 1.30
df['price_usd_infl_adj'] = df['price_usd'] * USD_INFLATION_FACTOR

# Maintain 'price' as nominal price_usd for backwards compatibility
df['price'] = df['price_usd']

print(f'After scope filtering & currency unification: {df.shape[0]:,} rows')
print(f' - Quoted in USD: {(~mask_ars).sum():,}')
print(f' - Converted from ARS (at 50 ARS/USD): {mask_ars.sum():,}')
print('\nProperty type breakdown:')
print(df['property_type'].value_counts())


After scope filtering & currency unification: 131,717 rows
 - Quoted in USD: 130,306
 - Converted from ARS (at 50 ARS/USD): 1,411

Property type breakdown:
property_type
Departamento    112005
PH               11454
Casa              8258
Name: count, dtype: int64


### 3. Location Standardization & Geographic Bounding Box
In the Properati hierarchy:
- `l1` = Country (`Argentina`)
- `l2` = Region / Federal District (`Capital Federal`)
- `l3` = **Barrio / Neighbourhood** (*Palermo, Belgrano, Caballito, etc.*)
- `l4` = Sub-barrio (*Palermo Soho, etc.* - ~96% NaN)

We also enforce the official geographic bounding box for Capital Federal:
- Latitude: `[-34.71, -34.52]`
- Longitude: `[-58.54, -58.33]`
This removes corrupted or mis-tagged coordinates that fall into other provinces or water bodies.


In [4]:
# Standardize location column names
df = df.rename(columns={
    'l1': 'country',
    'l2': 'region',
    'l3': 'barrio',
    'l4': 'sub_barrio'
})

# Filter by CABA bounding box
caba_lat = df['lat'].between(-34.71, -34.52)
caba_lon = df['lon'].between(-58.54, -58.33)
df = df[caba_lat & caba_lon].copy()

print(f'After CABA bounding box geocoding: {df.shape[0]:,} rows')
print('\nTop 10 Barrios represented:')
print(df['barrio'].value_counts(dropna=False).head(10))


After CABA bounding box geocoding: 118,925 rows

Top 10 Barrios represented:
barrio
Palermo          16636
Almagro          10083
Belgrano          9362
Villa Crespo      8443
Caballito         8433
Recoleta          5820
Villa Urquiza     4588
Barrio Norte      3513
Flores            3392
Balvanera         3374
Name: count, dtype: int64


### 4. Deduplication & Anti-Data Leakage Strategy
Real estate platforms suffer from heavy multi-agency duplicate listings and re-postings:
1. **Exact ID Duplicates**: Deduplicate by `id`.
2. **Content Duplicates**: Multi-realtor postings of identical units (`lat`, `lon`, `price_usd`, `surface_total`, `property_type`, `rooms`). If unhandled, identical units leak across train/test splits.


In [5]:
# 1. Deduplicate by unique listing ID
df = df.drop_duplicates(subset=['id'])

# 2. Deduplicate identical listings posted by different agencies
content_features = ['lat', 'lon', 'price_usd', 'surface_total', 'property_type', 'rooms']
df = df.drop_duplicates(subset=content_features)

print(f'After ID & content deduplication: {df.shape[0]:,} rows')


After ID & content deduplication: 79,124 rows


### 5. Physical Consistency & Missing Data Remediation
- **Numeric Coercion**: Ensure float/int conversions.
- **Surface Consistency**: Real-world property physics dictate $Surface_{total} \ge Surface_{covered}$. When reversed, swap the values. When one surface metric is absent, impute from the other.
- **Studio Apartments (*Monoambientes*)**: In Buenos Aires, 1-room units have `rooms == 1` and `bedrooms == NaN`. Imputing `bedrooms = 0` retains over 50% of studio apartments that would otherwise be discarded.


In [6]:
numeric_cols = [
    'lat', 'lon', 'rooms', 'bedrooms', 'bathrooms',
    'surface_total', 'surface_covered', 'price_usd', 'price_usd_infl_adj'
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Surface reconciliation
mask_total_null = df['surface_total'].isna() & df['surface_covered'].notna()
df.loc[mask_total_null, 'surface_total'] = df.loc[mask_total_null, 'surface_covered']

mask_cov_null = df['surface_covered'].isna() & df['surface_total'].notna()
df.loc[mask_cov_null, 'surface_covered'] = df.loc[mask_cov_null, 'surface_total']

# Enforce surface_total >= surface_covered
mask_inverted = (df['surface_total'] > 0) & (df['surface_covered'] > 0) & (df['surface_total'] < df['surface_covered'])
df.loc[mask_inverted, ['surface_total', 'surface_covered']] = df.loc[mask_inverted, ['surface_covered', 'surface_total']].values

# Impute studio apartments (monoambientes)
df.loc[(df['rooms'] == 1) & (df['bedrooms'].isna()), 'bedrooms'] = 0

print('Null counts after physical consistency checks:')
print(df[numeric_cols].isna().sum())


Null counts after physical consistency checks:
lat                       0
lon                       0
rooms                  6329
bedrooms              21797
bathrooms              5039
surface_total         14678
surface_covered       14678
price_usd                 0
price_usd_infl_adj        0
dtype: int64


### 6. Domain-Specific Market Fences (Outlier Handling)
Filter out extreme, physically impossible, or erroneous market entries:
- Total and covered surfaces: [15, 1000] m²
- Room counts: Rooms in [1, 10], Bathrooms in [1, 7]
- Total price: [$15,000, $3,000,000] USD
- Price per m²: [500, 8000] USD/m² (realistic market boundaries for CABA residential sales)


In [7]:
mask_valid = (
    df['surface_total'].between(15, 1000) &
    df['surface_covered'].between(15, 1000) &
    df['price_usd'].between(15_000, 3_000_000) &
    df['rooms'].between(1, 10) &
    df['bathrooms'].between(1, 7)
)
df = df[mask_valid].copy()

# Derived USD/m2 fences
df['price_usd_per_m2'] = df['price_usd'] / df['surface_total']
df = df[df['price_usd_per_m2'].between(500, 8000)].copy()

# Derived 30% inflation adjusted price per m2
df['price_usd_infl_adj_per_m2'] = df['price_usd_infl_adj'] / df['surface_total']
df['price'] = df['price_usd']

print(f'After domain outlier fencing: {df.shape[0]:,} rows')
print('\nSummary statistics of nominal vs. 30% inflation-adjusted prices:')
print(df[['price_usd', 'price_usd_infl_adj', 'price_usd_per_m2', 'price_usd_infl_adj_per_m2', 'surface_total', 'rooms', 'bedrooms']].describe().round(1))


After domain outlier fencing: 57,505 rows

Summary statistics of nominal vs. 30% inflation-adjusted prices:
       price_usd  price_usd_infl_adj  price_usd_per_m2  \
count    57505.0             57505.0           57505.0   
mean    237655.8            308952.5            2715.9   
std     239428.8            311257.4            1023.1   
min      22000.0             28600.0             500.0   
25%     115000.0            149500.0            2070.2   
50%     165000.0            214500.0            2583.3   
75%     265000.0            344500.0            3181.8   
max    3000000.0           3900000.0            8000.0   

       price_usd_infl_adj_per_m2  surface_total    rooms  bedrooms  
count                    57505.0        57505.0  57505.0   43424.0  
mean                      3530.7           89.3      2.7       1.6  
std                       1330.1           73.4      1.3       1.2  
min                        650.0           15.0      1.0       0.0  
25%                     

### 7. Feature Type Optimization & Redundant Column Cleanup
- Drop static / constant single-value columns (`country`, `region`, `operation_type`, `price_period`, empty `l5`, `l6`).
- Convert categoricals to `category` dtype for memory and modeling efficiency.
- Handle sentinel active-listing dates (`9999-12-31` -> `NaT`) to prevent PyArrow out-of-bounds timestamp exceptions.
- Parse date strings to `datetime64`.


In [8]:
redundant_cols = [
    col for col in [
        'country', 'region', 'operation_type', 'price_period', 'l5', 'l6', 'ad_type', 'currency'
    ] if col in df.columns
]
df = df.drop(columns=redundant_cols)

# Categorical columns
for col in ['property_type', 'barrio']:
    if col in df.columns:
        df[col] = df[col].astype('category')

# Handle sentinel active listing dates (9999-12-31)
if 'end_date' in df.columns:
    df.loc[df['end_date'].astype(str).str.startswith('9999'), 'end_date'] = pd.NaT

# Parse datetime columns
for col in ['start_date', 'end_date', 'created_on']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

print('Final columns:', list(df.columns))
print(f'Final memory usage: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB')


Final columns: ['id', 'start_date', 'end_date', 'created_on', 'lat', 'lon', 'barrio', 'sub_barrio', 'rooms', 'bedrooms', 'bathrooms', 'surface_total', 'surface_covered', 'price', 'title', 'description', 'property_type', 'price_usd', 'price_usd_infl_adj', 'price_usd_per_m2', 'price_usd_infl_adj_per_m2']
Final memory usage: 72.65 MB


### 8. Export Interim Cleaned Dataset
Export cleaned data in **Parquet format** (preserves data types, compression, and fast I/O for `02_eda_exploration.ipynb`) and **CSV format** (human-readable inspection fallback).


In [9]:
output_dir = Path('../data/interim')
output_dir.mkdir(parents=True, exist_ok=True)

parquet_output_path = output_dir / 'properties_cleaned.parquet'
df.to_parquet(parquet_output_path, index=False)
print(f'Saved primary interim dataset (Parquet): {parquet_output_path}')

csv_output_path = output_dir / 'properties_cleaned.csv'
df.to_csv(csv_output_path, index=False)
print(f'Saved inspection interim dataset (CSV): {csv_output_path}')

print(f'\nData Cleaning Completed Successfully!')
print(f'Total clean CABA listings ready for EDA/Modeling: {len(df):,}')


Saved primary interim dataset (Parquet): ..\data\interim\properties_cleaned.parquet
Saved inspection interim dataset (CSV): ..\data\interim\properties_cleaned.csv

Data Cleaning Completed Successfully!
Total clean CABA listings ready for EDA/Modeling: 57,505
